# Segment 1 Lab 2

## Making our own Customer Support Chatbot

One of the most common business use cases of Gen AI.

We'll even make our own User Interface - no frontend skills required!

We will use the delightful `gradio` framework which makes it remarkably easy for data scientists to build great UIs.

We are going to move quickly as this is just a teaser - but please come back and look at this later!

In [ ]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

In [ ]:
# Initialize

openai = OpenAI()
MODEL = 'gpt-5.6-luna'

In [ ]:
system_prompt = """
You are a customer support assistant for an airline.
You give short, humorous, snarky answers, no more than 2-3 sentences.
For context, if it's relevant: a return ticket to London costs $499.
"""

system_message = {"role": "system", "content": system_prompt}

Reminder of the structure of prompt messages to OpenAI:

```
[
    {"role": "system", "content": "system message here"},
    {"role": "user", "content": "first user prompt here"},
    {"role": "assistant", "content": "the assistant's response"},
    {"role": "user", "content": "the new user prompt"},
]
```



## The chat function

In order to use Gradio's out-of-the-box Chat User Interface, we need to write a single function, `chat(message, history)`

In [ ]:
def chat(message, history):
    messages = [system_message] + history + [{"role": "user", "content": message}]
    results = openai.chat.completions.create(model=MODEL, messages=messages, reasoning_effort="none")
    return results.choices[0].message.content

## And then enter Gradio's magic!

In [ ]:
gr.ChatInterface(chat, type="messages").launch()

# Let's go multi-modal!!

We can use gpt-image-2 to make us some images.

Let's put this in a function called artist.

### Price alert: each time I generate an image it costs about 4c - don't go crazy with images!

In [ ]:
# Some imports for handling images

import base64
from io import BytesIO
from PIL import Image

In [ ]:
def artist(city):
    prompt = f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}, in a vibrant pop-art style"
    image_response = openai.images.generate(
            model="gpt-image-2",
            prompt=prompt,
            size="1024x1024",
            n=1,
            quality="low",
        )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))

In [ ]:
image = artist("New York City")
display(image)

# Bringing it together

This is the start of what you might call an "agent framework", in that we will use multiple LLM calls to solve a complex problem.

We'll work on a full agent framework in the final project today!

In [ ]:
def chat(history):
    message = history[-1]["content"]
    messages = [system_message] + history
    results = openai.chat.completions.create(model=MODEL, messages=messages)
    image = artist("London") if "london" in message.lower() else None
    response = results.choices[0].message.content
    history += [{"role":"assistant", "content":response}]
    return history, image

In [ ]:
# More involved Gradio code as we're not using the preset Chat interface

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=360, type="messages")
        image_output = gr.Image(height=360)
    with gr.Row():
        entry = gr.Textbox(label="Chat with our AI Assistant:")

    def do_entry(message, history):
        history += [{"role":"user", "content":message}]
        return "", history

    entry.submit(do_entry, inputs=[entry, chatbot], outputs=[entry, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, image_output]
    )

ui.launch()

# A multi-modal customer support chatbot - in minutes!

This illustrated how easy it is to build a chatbot with character and knowledge.

# Exercise

Take this further - have it generate audio for its responses, and use Tools to look up costs of flights.  
See my companion repo llm_engineering in week2 folder for the solution.

## And.. apply this to your business! Make an AI Assistant for your domain